# Cricket Intelligence System
## Notebook 1 - Data Cleaning

This notebook loads all 9 raw CSV files across ODI, Test, and T20 formats
(batting, bowling, fielding), cleans them, and saves 3 master cleaned CSVs.

### Steps:
1. Import libraries
2. Load all 9 CSVs
3. Clean each file (drop unnamed columns, extract country, parse span, clean HS)
4. Add format column
5. Stack all formats into 3 master dataframes
6. Save to data/cleaned/

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# Base paths
RAW = 'C:/cricket_intelligence/data/raw/'
CLEANED = 'C:/cricket_intelligence/data/cleaned/'

# File paths for all 9 CSVs
files = {
    'batting': {
        'ODI': RAW + 'ODI_data.csv',
        'Test': RAW + 'test.csv',
        'T20': RAW + 't20.csv'
    },
    'bowling': {
        'ODI': RAW + 'Bowling_ODI.csv',
        'Test': RAW + 'Bowling_test.csv',
        'T20': RAW + 'Bowling_t20.csv'
    },
    'fielding': {
        'ODI': RAW + 'Fielding_ODI.csv',
        'Test': RAW + 'Fielding_test.csv',
        'T20': RAW + 'Fielding_t20.csv'
    }
}

In [7]:
# Load all 9 CSVs into a nested dictionary
dfs = {}

for discipline, formats in files.items():
    dfs[discipline] = {}
    for format_name, path in formats.items():
        dfs[discipline][format_name] = pd.read_csv(path)
        print(f"Loaded {discipline} - {format_name}: {dfs[discipline][format_name].shape}")

Loaded batting - ODI: (2500, 15)
Loaded batting - Test: (3001, 13)
Loaded batting - T20: (2006, 17)
Loaded bowling - ODI: (2582, 15)
Loaded bowling - Test: (3050, 16)
Loaded bowling - T20: (2006, 16)
Loaded fielding - ODI: (2600, 13)
Loaded fielding - Test: (3001, 13)
Loaded fielding - T20: (2006, 13)


In [4]:
import os
print(os.getcwd())

C:\cricket_intelligence\notebooks


In [6]:
print(os.listdir('C:/cricket_intelligence/data/raw/'))

['Bowling_ODI.csv', 'Bowling_t20.csv', 'Bowling_test.csv', 'Fielding_ODI.csv', 'Fielding_t20.csv', 'Fielding_test.csv', 'ODI_data.csv', 't20.csv', 'test.csv']


In [8]:
def basic_clean(df, format_name, discipline):
    # Drop unnamed columns
    df = df.drop(columns=[col for col in df.columns if 'Unnamed' in col])
    
    # Extract country from player name e.g. 'SR Tendulkar (INDIA)' -> 'INDIA'
    df['Country'] = df['Player'].str.extract(r'\(([^)]+)\)')
    
    # Clean player name - remove country part
    df['Player'] = df['Player'].str.replace(r'\s*\(.*?\)', '', regex=True).str.strip()
    
    # Parse Span into start year, end year, career length
    df['Start_Year'] = df['Span'].str.split('-').str[0].astype(int)
    df['End_Year'] = df['Span'].str.split('-').str[1].astype(int)
    df['Career_Length'] = df['End_Year'] - df['Start_Year'] + 1
    
    # Add format and discipline columns
    df['Format'] = format_name
    df['Discipline'] = discipline
    
    return df

In [9]:
for discipline, formats in dfs.items():
    for format_name, df in formats.items():
        dfs[discipline][format_name] = basic_clean(df, format_name, discipline)
        print(f"Cleaned {discipline} - {format_name}: {dfs[discipline][format_name].shape}")

Cleaned batting - ODI: (2500, 19)
Cleaned batting - Test: (3001, 17)
Cleaned batting - T20: (2006, 21)
Cleaned bowling - ODI: (2582, 19)
Cleaned bowling - Test: (3050, 20)
Cleaned bowling - T20: (2006, 20)
Cleaned fielding - ODI: (2600, 17)
Cleaned fielding - Test: (3001, 17)
Cleaned fielding - T20: (2006, 17)


In [10]:
# Clean HS column in batting files only
for format_name in dfs['batting']:
    dfs['batting'][format_name]['HS'] = (
        dfs['batting'][format_name]['HS']
        .astype(str)
        .str.replace('*', '', regex=False)
        .replace('-', np.nan)
        .astype(float)
    )

print("HS column cleaned for all batting formats")

HS column cleaned for all batting formats


In [11]:
# Clean bowling numeric columns - replace '-' with NaN
bowling_numeric = ['Ave', 'Econ', 'SR']

for format_name in dfs['bowling']:
    for col in bowling_numeric:
        dfs['bowling'][format_name][col] = (
            pd.to_numeric(dfs['bowling'][format_name][col], errors='coerce')
        )

print("Bowling numeric columns cleaned for all formats")

Bowling numeric columns cleaned for all formats


In [12]:
# Stack all formats into 3 master dataframes
batting_master = pd.concat([dfs['batting']['ODI'], 
                            dfs['batting']['Test'], 
                            dfs['batting']['T20']], 
                            ignore_index=True)

bowling_master = pd.concat([dfs['bowling']['ODI'], 
                            dfs['bowling']['Test'], 
                            dfs['bowling']['T20']], 
                            ignore_index=True)

fielding_master = pd.concat([dfs['fielding']['ODI'], 
                             dfs['fielding']['Test'], 
                             dfs['fielding']['T20']], 
                             ignore_index=True)

print(f"Batting master: {batting_master.shape}")
print(f"Bowling master: {bowling_master.shape}")
print(f"Fielding master: {fielding_master.shape}")

Batting master: (7507, 21)
Bowling master: (7638, 23)
Fielding master: (7607, 17)


In [13]:
batting_master.to_csv('C:/cricket_intelligence/data/cleaned/batting_master.csv', index=False)
bowling_master.to_csv('C:/cricket_intelligence/data/cleaned/bowling_master.csv', index=False)
fielding_master.to_csv('C:/cricket_intelligence/data/cleaned/fielding_master.csv', index=False)

print("Saved all 3 master CSVs to data/cleaned/")

Saved all 3 master CSVs to data/cleaned/


In [14]:
# Quick sanity check on all 3 master files
for name, df in [('Batting', batting_master), 
                 ('Bowling', bowling_master), 
                 ('Fielding', fielding_master)]:
    print(f"=== {name} ===")
    print(f"Shape: {df.shape}")
    print(f"Formats: {df['Format'].value_counts().to_dict()}")
    print(f"Null counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print()

=== Batting ===
Shape: (7507, 21)
Formats: {'Test': 3001, 'ODI': 2500, 'T20': 2006}
Null counts:
HS          182
BF         3001
SR         3001
Country      39
4s         5501
6s         5501
dtype: int64

=== Bowling ===
Shape: (7638, 23)
Formats: {'Test': 3050, 'ODI': 2582, 'T20': 2006}
Null counts:
Balls      2006
Ave        1851
Econ       1360
SR         1851
4          3050
Country      39
BBM        4588
10         4588
Overs      5632
Mdns       5632
dtype: int64

=== Fielding ===
Shape: (7607, 17)
Formats: {'Test': 3001, 'ODI': 2600, 'T20': 2006}
Null counts:
Country    39
dtype: int64

